# CA2 – Online Retail: Sales Analysis and Interactive Dashboard

**Module:** Data Visualisation and Communication &nbsp;|&nbsp; **Lecturer:** Taufique Ahmed  
**Student:** Matheus Vinicios de Castilho &nbsp;|&nbsp; **Student number:** 2023336

**GitHub repository:** https://github.com/Matheus-Castilho/CA2-DV 

> **How to run this notebook:** place `Online Retail.xlsx` in the same folder as this notebook (the data is not included in the repository), then select *Kernel → Restart Kernel and Run All Cells*.  
> Libraries used: `pandas`, `matplotlib`, `openpyxl` (to read Excel) and `ipywidgets` (for the dashboard).

<a id="intro"></a>
## 1. Introduction

 #TODO

<a id="setup"></a>
## 2. Setup
mporting the libraries used (pandas for data handling, matplotlib for charts) and setting a consistent chart style

In [17]:
import pandas as pd
import matplotlib.pyplot as plt

# Show all columns and format decimals neatly
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Same chart style across the whole notebook
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

<a id="load"></a>
## 3. Loading the data

`InvoiceNo` and `StockCode` are read as **text** because some of their values contain letters (for example `C536379` or `85123A`). Without this, pandas would mix numbers and text in the same column.

The original data is kept unchanged in `df_raw`, and all cleaning is done on a copy called `df`. This makes it possible to compare before and after cleaning. Loading the Excel file can take around 30 seconds.

In [18]:
# Read InvoiceNo and StockCode as text because some values contain letters
df_raw = pd.read_excel('Online Retail.xlsx', dtype={'InvoiceNo': str, 'StockCode': str})

# Work on a copy so the original data stays unchanged
df = df_raw.copy()

print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head()

Rows: 541,909  |  Columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


<a id="quality"></a>
## 4. Data quality assessment

Before cleaning, the data is checked for problems: wrong data types, missing values, duplicates, invalid records (such as cancellations and negative quantities), items that are not real products, and outliers. **Nothing is removed in this section.** Each issue is only measured here, and the decisions about how to handle it are taken in the cleaning section.

### 4.1 Structure and data types

The table below describes each column and the data type it should have. `df.info()` is then used to compare this with how pandas actually read the data.

| Column | Meaning | Expected type |
|---|---|---|
| `InvoiceNo` | Invoice number. Invoices starting with **C** are cancellations | Text |
| `StockCode` | Product code | Text |
| `Description` | Product name | Text |
| `Quantity` | Units of the product on this invoice line | Whole number |
| `InvoiceDate` | Date and time of the transaction | Date/time |
| `UnitPrice` | Price per unit in pounds sterling (£) | Decimal |
| `CustomerID` | Customer number | ID (text) |
| `Country` | Country where the customer is based | Text |

In [19]:
# Column names, data types and number of non-missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(3)
memory usage: 45.7+ MB


**Observations:**
- The dataset has **541,909 rows and 8 columns**.
- `InvoiceDate` is already read as a date/time, so sales can be grouped by month, weekday and hour without any conversion.
- `CustomerID` is stored as a decimal number (`float64`). This happens because the column has missing values. It is an identifier, not something to calculate with, so it will be converted to text during cleaning.
- `Description` and `CustomerID` have fewer non-null values than the other columns, so they contain missing data. This is checked in the next section.

### 4.2 Descriptive statistics

Summary statistics (count, mean, median, minimum and maximum) give a quick overview of the numeric columns and highlight values that cannot be correct. Counting the unique values in each column shows how many invoices, products, customers and countries the data covers.

In [20]:
# Summary statistics for the numeric and date columns
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,"541,909.00",541909,"541,909.00","406,829.00"
mean,9.55,2011-07-04 13:34:57.156386,4.61,"15,287.69"
min,"-80,995.00",2010-12-01 08:26:00,"-11,062.06","12,346.00"
25%,1.00,2011-03-28 11:34:00,1.25,"13,953.00"
50%,3.00,2011-07-19 17:17:00,2.08,"15,152.00"
75%,10.00,2011-10-19 11:27:00,4.13,"16,791.00"
max,"80,995.00",2011-12-09 12:50:00,"38,970.00","18,287.00"
std,218.08,NaN,96.76,"1,713.60"


In [21]:
# Number of different values in each column
df.nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

**Observations:**
- `Quantity` ranges from **−80,995 to +80,995**. Both extremes are the same size, which suggests one very large order that was entered and then cancelled. This is checked in the outliers section.
- `UnitPrice` has a **negative minimum (−£11,062.06)**, which cannot be a real price. Rows like this are accounting adjustments, not sales.
- The **median** quantity is 3 units and the median price is about £2, but both **means** are higher. This shows the data is **right-skewed**: most invoice lines are small, with a few very large ones pulling the average up. For this reason the median is a better measure of a "typical" order line.
- The data covers about **25,900 invoices, 4,070 products, 4,372 customers and 38 countries**, from December 2010 to December 2011.
- `Description` has more unique values than `StockCode` (4,223 vs 4,070), so some products appear under slightly different names. To avoid splitting one product into several rows, product charts will group by `StockCode` and show one name per code.